<a href="https://colab.research.google.com/github/SANGRAMLEMBE/Computer_Vision_from_Scratch/blob/main/Overfitting/4_overfitting_with_all_4_technics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  https://wandb.ai/sangram01-srm-institute-of-science-and-technology/5_flower_simple_neural_network?nw=nwusersangram01

In [ ]:
import wandb
from wandb.integration.keras import WandbMetricsLogger , WandbModelCheckpoint


import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras

In [ ]:
sweep_config = {
    'method': 'grid',
    'metric': {
      'name': 'val_accuracy',
      'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {'values': [0.001] },
        'batch_size': {'values': [8] },
        'hidden_nodes': {'values': [128] },
        'img_size':{'values':[16]},
        'epochs':{'values':[10]}
    }
}
sweep_id = wandb.sweep(sweep_config, project="5_flower_neural_network_with_L2_regularization")



In [ ]:
# from here we can access parameters

def train():
  with wandb.init() as run:
    config = wandb.config

    IMG_HEIGHT = config.img_size
    IMG_WIDTH = config.img_size
    IMG_CHANNELS = 3
    CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]



    def read_and_decode(filename, resize_dims):

      img_bytes = tf.io.read_file(filename)
      img = tf.image.decode_jpeg(img_bytes, channels = IMG_CHANNELS)
      img = tf.image.convert_image_dtype(img, tf.float32)
      img = tf.image.resize(img,resize_dims)

      return img

    def parse_csvline(csv_line):

      # record_default specify the data types for each column
      record_default = ["",""]
      filename, label_string = tf.io.decode_csv(csv_line, record_default)

      #Load the Image
      img  = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])

      # Convert label string to integer based on the CLASS_NAMES index
      label = tf.argmax(tf.math.equal(CLASS_NAMES, label_string))


      return img, label


    # Define datasets
    train_dataset = (
        tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/train_set.csv")
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
      )          # we defined batch size = 16

    eval_dataset = (
        tf.data.TextLineDataset("gs://practical-ml-vision-book-data/flowers_5_jpeg/flower_photos/eval_set.csv")
        .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(config.batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )




    import matplotlib.pyplot as plt

    for image_batch, label_batch in train_dataset.take(2):

      #Take the first image from the batch
      first_image = image_batch[0]
      first_label = label_batch[0]


      # Conver Tensor To Numpy array
      plt.imshow(first_image.numpy())
      plt.title(f"Label:{CLASS_NAMES[first_label]}")
      plt.axis('off')
      plt.show()


    # Defining L2 Regularization
    regularizer = tf.keras.regularizers.l1_l2(0, 0.1)

    model= keras.Sequential([

        keras.layers.Flatten(input_shape = (IMG_HEIGHT, IMG_WIDTH,
                                            IMG_CHANNELS)),

        keras.layers.Dense(config.hidden_nodes, kernal_regularizer = regularizer),
        keras.layers.BatchNormalization(),
        keras.layers.activation( "relu"),
        keras.layers.Dropout(0.5),                                                    # Dropouts


        keras.layers.Dense(len(CLASS_NAMES), kernal_regularizer = regularizer),         # Regularization
        keras.layers.BatchNormalization(),                                             # Batch Normalization
        keras.layers.Activation("softmax")
    ])

    model.compile(
        optimizer = "adam",
        loss = keras.losses.SparseCategoricalCrossentropy(from_logits = False),
        metrics = ["accuracy"]
    )

    model.fit(
        train_dataset,
        validation_data = eval_dataset,
        epochs = config.epochs,

        callbacks = [
            WandbMetricsLogger(log_freq=5), # this is for Logging your training o/p on W&B website

            tf.keras.callbacks.EarlyStopping(monitor = "val_loss",
                                             patience = 2, restore_best_weights = True)  # Early Stopping


            ]
    )




In [ ]:
wandb.agent(sweep_id, function=train)